## Bronze Layer - Final Table Created

**First step:** create the final table `documents_parsed_classified`, which classifies each file as an invoice, a purchase order, or a receipt. Then, based on each document's type, extract the necessary data.

In [0]:
CREATE OR REPLACE TABLE parsed_documents
SELECT *,
ai_parse_document(content) AS parsed_content,
parsed_content:document:elements AS elements,
SIZE(CAST(elements AS ARRAY<VARIANT>)) AS element_count
FROM read_files(
    '/Volumes/medical_finance/bronze_layer/raw_files',
    format => 'binaryFile'
)

In [0]:
SELECT *
FROM parsed_documents 

In [0]:
-- not very necessary step , aims to show the content in a pretty human-readable way
CREATE OR REPLACE TABLE parsed_documents_pretty
SELECT *,
concat_ws('\n',
    transform(cast(elements as array<variant>), -- an array containing only the content of each element
        element -> element:content
    )
) AS element_pretty_format
FROM parsed_documents

In [0]:
SELECT * 
FROM parsed_documents_pretty

In [0]:
CREATE OR REPLACE TABLE classification_results
SELECT path ,
parsed_content,
element_pretty_format,
element_count,
ai_classify(
    parsed_content,
    '["Invoice", "Purchase Order","Receipt"]',
    map("version", "2.0")
) AS calssification_result,
calssification_result:response[0]::STRING AS document_type
FROM parsed_documents_pretty

In [0]:
-- now it is time to put final parsed and classified data in a table at bronze level
CREATE OR REPLACE TABLE medical_finance.bronze_layer.documents_parsed_classified
SELECT 
  path,
  parsed_content,
  element_pretty_format,
  document_type
FROM   classification_results

In [0]:
SELECT *
FROM medical_finance.bronze_layer.documents_parsed_classified

##  Extracting Structured Data

**Second step:** extract the structured data from each document and save it in the **silver** schema of our catalog, with one table per document type: invoices, purchase orders, and receipts.

### 1. Invoices

In [0]:
CREATE OR REPLACE TABLE medical_finance.bronze_layer.invoices
WITH invoices AS
(
SELECT
  path,
  element_pretty_format,
  ai_extract(
    parsed_content,
    '["invoice_number", "invoice_date", "po_number", "seller_email", "shipping_address","payment_method",
      "buyer", "seller", "currency", "total_amount"]',
    map('version', '2.1', 'instructions', 'These are invoices.')
  ) AS extracted_info
FROM medical_finance.bronze_layer.documents_parsed_classified
WHERE document_type = 'Invoice'
)
SELECT
 path,
 element_pretty_format AS full_invoice_info,
 extracted_info:response:invoice_number:value::STRING AS invoice_number,
 extracted_info:response:invoice_date:value::STRING AS invoice_date,
 extracted_info:response:po_number:value::STRING AS po_number,
 extracted_info:response:seller:value::STRING AS seller,
 extracted_info:response:seller_email:value::STRING AS seller_email,
 extracted_info:response:buyer:value::STRING AS buyer,
 extracted_info:response:shipping_address:value::STRING AS shipping_address,
 extracted_info:response:payment_method:value::STRING AS payment_method,
 extracted_info:response:currency:value::STRING AS currency,
 extracted_info:response:total_amount:value::STRING AS total_amount
FROM invoices
-- now we have all the data we need in a table at bronze level
-- we can use this table to create silver tables for each document type

In [0]:
CREATE OR REPLACE TABLE medical_finance.bronze_layer.purchase_orders
WITH purchase_orders AS
(
SELECT
  path,
  element_pretty_format,
  ai_extract(
    parsed_content,
    '["po_number", "po_date", "requested_ship_date", "buyer", "vendor", "currency", "subtotal", "shipping", "total_amount"]',
    map('version', '2.1', 'instructions', 'These are purchase orders. If a field is missing, return null.')
  ) AS extracted_info
FROM medical_finance.bronze_layer.documents_parsed_classified
WHERE document_type = 'Purchase Order'
)
SELECT
 path,
 element_pretty_format AS full_po_info,
 extracted_info:response:po_number:value::STRING AS po_number,
 extracted_info:response:po_date:value::STRING AS po_date,
 extracted_info:response:requested_ship_date:value::STRING AS requested_ship_date,
 extracted_info:response:buyer:value::STRING AS buyer,
 extracted_info:response:vendor:value::STRING AS vendor,
 extracted_info:response:currency:value::STRING AS currency,
 extracted_info:response:subtotal:value::STRING AS subtotal,
 extracted_info:response:shipping:value::STRING AS shipping,
 extracted_info:response:total_amount:value::STRING AS total_amount
FROM purchase_orders;


### 3. Receipt

In [0]:
CREATE OR REPLACE TABLE medical_finance.bronze_layer.receipts
WITH Receipts AS (
    SELECT 
        *,
        ai_extract(parsed_content,
        '["Receipt_number", "vendor", "datetime", "currency","total_amount"]',
        map('version', '2.1', 'instructions', 'These are receipts.')
        ) AS extracted_info
    FROM medical_finance.bronze_layer.documents_parsed_classified
    WHERE document_type = 'Receipt'
)

SELECT 
    path,
    element_pretty_format AS full_receipt_info,
    extracted_info:response:Receipt_number:value::STRING AS Receipt_number,
    extracted_info:response:vendor:value::STRING AS vendor,
    extracted_info:response:datetime:value::STRING AS datetime,
    extracted_info:response:currency:value::STRING AS currency,
    extracted_info:response:total_amount:value::STRING AS total_amount
FROM Receipts

### 4. Invoice Items

In [0]:
CREATE OR REPLACE TABLE medical_finance.bronze_layer.invoice_items
WITH invoice_items AS (
  SELECT
    path,
    ai_extract(
      parsed_content,
      '{
        "invoice_number": {"type": "string"},
        "line_items": {
          "type": "array",
          "description": "All individual line items listed in the invoice",
          "items": {
            "type": "object",
            "properties": {
              "description": {"type": "string", "description": "Product or service description"},
              "qty":          {"type": "number", "description": "Quantity"},
              "unit_price":   {"type": "number", "description": "Price per unit"},
              "amount":       {"type": "number", "description": "Line total amount"}
            }
          }
        }
      }',
      MAP('version', '2.1', 'instructions', 'These are medical/finance invoices. Extract every line item row from each invoice.')
    ) AS extracted
  FROM medical_finance.bronze_layer.documents_parsed_classified
  WHERE document_type = 'Invoice'
)
SELECT
  path,
  extracted:response:invoice_number:value::STRING AS invoice_number,
  item:description:value::STRING  AS description,
  item:qty:value::DOUBLE          AS qty,
  item:unit_price:value::DOUBLE   AS unit_price,
  item:amount:value::DOUBLE       AS amount
FROM invoice_items
LATERAL VIEW EXPLODE(
  CAST(extracted:response:line_items AS ARRAY<VARIANT>)
) exploded AS item;